# ภาพย้อนหลังสามพื้นที่และตำแหน่งดาวเทียม
ใช้ source folder และ research/ จากโปรเจกต์นี้ รันด้วย environment ใน `.venv` หรือวาง source บน Drive แล้วกำหนด `EO_DEMO_ROOT` ใน Colab ติดตั้ง `requirements-lock.txt` ใน kernel ก่อนเริ่ม ไม่จำเป็นต้องโหลด SICD/CPHD ทั้งไฟล์
ผลตรวจจับในโน้ตบุ๊กนี้เป็น bright candidates ที่ยังไม่ผ่านการยืนยันว่าเป็นรถหรือเรือ

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path(os.environ.get('EO_DEMO_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'eo_engine').exists(), 'Set EO_DEMO_ROOT to the Earth Observation Demo source directory (local checkout or mounted Drive).'
sys.path.insert(0, str(ROOT))


In [ ]:
from eo_engine.catalog import scenes, scene_dir
from eo_engine.pipeline import prepare, detect
from IPython.display import display, Image
rows = scenes()
print({site: sum(r['site_id'] == site for r in rows) for site in ('suvarnabhumi','rotterdam','singapore')})


In [ ]:
selected = [next(r for r in rows if r['site_id'] == site) for site in ('suvarnabhumi','rotterdam','singapore')]
for row in selected:
    if not row['ready']: prepare(row['id'], width_m=250)
    result = detect(row['id'])
    print(row['site_id'], row['center_utc'], row['platform'], 'bright candidates:', result['count'])
    display(Image(filename=str(scene_dir(row['id']) / 'preview.png')))


In [ ]:
import json
for row in selected:
    g = json.loads((scene_dir(row['id']) / 'geometry.json').read_text())
    p = g['samples'][len(g['samples'])//2]
    print(row['site_id'], {'source':g['source'], 'altitude_km':p['altitude_m']/1000, 'azimuth_deg':p['azimuth_deg'], 'elevation_deg':p['elevation_deg'], 'metadata_bytes_read':g['downloaded_bytes']})


## ตรวจผล
ตรวจ ROI และแยก shadow/layover ก่อนตีความผล Baseline นี้ไม่มี road/water semantic mask หรือ precision/recall ที่ตรวจแล้ว ใช้แอปช่วย Accept/Reject และ export CSV ได้